### This Notebook code fixes the issue with the `docling_layout_reading_order_sample.pdf` 
section 1 table, where we do have Ingestion Layer and Analytics Layer table. This table is a borderless table
technically and Docling failed to read this section Analytics Layer column data in the section 1 itself. Docling has moved this section 1 text to section 3( that has been found in output directory .md file for reference). That the reading order has been changed/missed.

#### The below code fixes it

### Below code is useful to check whether the PDF file exist or not

In [ ]:
from pathlib import Path

# point to the PDF file path you want to convert
pdf_path = Path(
    "D:/AI Learning/rag-learning/data/raw/pdf/docling_layout_reading_order_sample.pdf"
)

print(pdf_path.exists()) # checks if the file exists
print(pdf_path) # prints the path to the PDF file

In [ ]:
from pathlib import Path

from docling.datamodel.base_models import InputFormat

from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    TableFormerMode,
    TableStructureOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption

# 1. Configure the PDF Pipeline Options
pipeline_options = PdfPipelineOptions()

# Enable deep table structure extraction
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options = TableStructureOptions(
    mode=TableFormerMode.ACCURATE,  # Use full structural prediction model
    do_cell_matching=True,  # Force cell alignment across borderless columns
)

# Optional: Generate page images for visual debugging if needed
pipeline_options.generate_page_images = False

# 2. Initialize Converter with the configured PDF options
converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

# 3. Convert the document
pdf_path = Path(
    "D:/AI Learning/rag-learning/data/raw/pdf/docling_layout_reading_order_sample.pdf"
)
result = converter.convert(pdf_path)
doc = result.document

# 4. Export to Markdown
markdown_text = doc.export_to_markdown()
print(markdown_text)

# 5. Save the output
output_path = Path("output/stage1.4.2.5.3_corrected_docling.md")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(markdown_text, encoding="utf-8")

print(f"\nSuccessfully parsed and saved: {output_path}")

### Below converts the PDF into a Docling Document reads it and preserves the reading order.

In [6]:
from pathlib import Path
# Docling imports to convert PDF to Docling document and manipulate the reading order
from docling.document_converter import DocumentConverter

# 1. Convert PDF
converter = DocumentConverter()
pdf_path = Path(
    "D:/AI Learning/rag-learning/data/raw/pdf/docling_layout_reading_order_sample.pdf"
)
result = converter.convert(pdf_path)
doc = result.document

# Helper function to resolve JSON pointer reference (e.g. "#/texts/4" -> doc.texts[4])
def resolve_item(doc_obj, child_ref):
    ref_str = (
        getattr(child_ref, "cref", None)
        or getattr(child_ref, "ref", None)
        or getattr(child_ref, "$ref", None)
    )
    if ref_str and isinstance(ref_str, str) and ref_str.startswith("#/"):
        parts = ref_str.lstrip("#/").split("/")
        category, idx = parts[0], int(parts[1])
        return getattr(doc_obj, category)[idx]
    return child_ref

# 2. Extract items with their bounding box metadata
page_1_items = []
other_page_items = []

for child in doc.body.children:
    item = resolve_item(doc, child)
    
    # Check item provenance
    if hasattr(item, "prov") and item.prov:
        prov = item.prov[0]
        page_no = prov.page_no
        bbox = prov.bbox
        
        if page_no == 1:
            page_1_items.append((bbox.t, bbox.l, bbox.r, child, item))
        else:
            other_page_items.append(child)
    else:
        other_page_items.append(child)

# 3. Separate Page 1 into sections
# Top content: Overview & Intro (top > 620)
# Multi-column block: Ingestion & Analytics (top between 470 and 620)
# Bottom content: Processing Stages & Design Considerations (top <= 470)
top_items = []
left_col_items = []
right_col_items = []
bottom_items = []

for t, l, r, child, item in page_1_items:
    if t > 620:
        top_items.append((t, l, child))
    elif 470 <= t <= 620:
        if l < 290:  # Left column (Ingestion Layer)
            left_col_items.append((t, l, child))
        else:        # Right column (Analytics Layer)
            right_col_items.append((t, l, child))
    else:
        bottom_items.append((t, l, child))

# Sort each subsection by vertical coordinate descending (top-to-bottom)
top_sorted = [c for _, _, c in sorted(top_items, key=lambda x: -x[0])]
left_sorted = [c for _, _, c in sorted(left_col_items, key=lambda x: -x[0])]
right_sorted = [c for _, _, c in sorted(right_col_items, key=lambda x: -x[0])]
bottom_sorted = [c for _, _, c in sorted(bottom_items, key=lambda x: -x[0])]

# 4. Reconstruct body sequence: Overview -> Ingestion Layer -> Analytics Layer -> Stages -> Design
doc.body.children = (
    top_sorted
    + left_sorted
    + right_sorted
    + bottom_sorted
    + other_page_items
)

# 5. Export and save corrected Markdown
markdown_text = doc.export_to_markdown()
print(markdown_text)

# output path to save the corrected markdown
output_path = Path("output/stage1.4.2.5.3_fixed_reading_order.md")
output_path.parent.mkdir(parents=True, exist_ok=True) # ignore if the directory already exists
output_path.write_text(markdown_text, encoding="utf-8")

print(f"\nSaved corrected markdown to: {output_path}")

[INFO] 2026-08-20 13:41:05,230 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-20 13:41:05,232 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-20 13:41:05,281 [RapidOCR] download_file.py:60: File exists and is valid: D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-20 13:41:05,282 [RapidOCR] main.py:50: Using D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-08-20 13:41:05,642 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-20 13:41:05,645 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-20 13:41:05,651 [RapidOCR] download_file.py:60: File exists and is valid: D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-08-20 13:41:05,652 [RapidOCR] main.py:50: Using D:\AI Learning\rag-learning\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobil

## Azure Event Processing Architecture

This sample document is designed for Stage 1.4.2.5.3 - Docling Layout &amp; Reading Order. It contains headings, paragraphs, a two-column layout, a table, a caption, and content whose logical reading order differs from simple top-to-bottom visual scanning.

## 1. Overview

An enterprise application publishes events to Azure Event Hubs. A downstream processing component consumes those events and sends selected records to Azure Data Explorer for analytics. The architecture contains independent ingestion and analytics stages.

## Ingestion Layer

The producer sends events to Event Hubs. Event Hubs provides scalable event ingestion and partitions the event stream for parallel consumption.

Key responsibility: reliably accept high-volume event data.

Figure 1 - Logical processing flow

## Analytics Layer

A consumer reads events and writes analytical records into Azure Data Explorer. Queries can then be used for operational monitoring and historical a